# Formulación QUBO: Optimización de Carteras (Portfolio Optimization)

En este notebook aprenderemos a formular y resolver uno de los problemas canónicos de optimización combinatoria en finanzas: la **selección óptima de una cartera de activos** (inspirado en el modelo clásico de Markowitz).

### Objetivos de aprendizaje
1. **Comprender el problema de negocio:** Balancear el rendimiento esperado de una inversión frente al riesgo (covarianza) bajo una restricción de presupuesto.
2. **De problema restringido a QUBO:** Convertir una función objetivo con restricciones de desigualdad/igualdad a una función de coste sin restricciones (*Unconstrained*) mediante penalizaciones cuadráticas.
3. **Modelado simbólico con PyQUBO:** Usar herramientas de alto nivel (`Array`, `Constraint`, `Placeholder`) para no tener que construir la matriz de interacción a mano.
4. **Impacto de los hiperparámetros:** Analizar cómo los multiplicadores de Lagrange ($	heta_1, 	heta_2, 	heta_3$) determinan si la solución prioriza el beneficio, la diversificación o el cumplimiento estricto del presupuesto.
5. **Resolución clásica local:** Resolver el problema con cálculo exacto (`dimod.ExactSolver`) y con recocido simulado clásico (`neal.SimulatedAnnealingSampler`), sin requerir acceso físico a una QPU.


In [ ]:
# Instalación de librerías necesarias (si ejecutas en Google Colab o un entorno nuevo)
!pip install -q pyqubo dimod dwave-neal numpy matplotlib tabulate

## 1. Formulación matemática clásica (Markowitz)

Supongamos que disponemos de un universo de $n$ activos financieros posibles. Para cada activo $i$, conocemos:
* $r_i$: El rendimiento o beneficio esperado al invertir en el activo $i$.
* $b_i$: El coste de capital o inversión requerida para adquirir el activo $i$.
* $c_{ij}$: La covarianza entre los activos $i$ y $j$, que cuantifica el riesgo conjunto (si dos activos tienden a caer al mismo tiempo, coinvertir en ambos aumenta el riesgo).
* $B$: Nuestro presupuesto total disponible.

Definimos las variables de decisión binarias:
$$x_i \in \{0, 1\} \quad 	\text{donde } x_i = 1 	\text{ si incluimos el activo } i 	\text{ en la cartera, y } 0 	\text{ en caso contrario.}$$

El problema clásico de optimización se enuncia como:
$$\max_{x} \left[ \sum_{i=1}^{n} r_i x_i - \sum_{i=1}^{n} \sum_{j > i} c_{ij} x_i x_j \right]$$
$$\text{sujeto a } \sum_{i=1}^{n} b_i x_i \le B$$

## 2. Traducción al formato QUBO

Un procesador cuántico o un algoritmo de annealing no acepta restricciones independientes ni funciones a maximizar. Requiere un formato **QUBO** (*Quadratic Unconstrained Binary Optimization*), formulado siempre como **minimización sin restricciones**:

$$\min_{x \in \{0, 1\}^n} \mathcal{H}(x)$$

Para transformar nuestro problema:
1. **Invertimos el retorno:** Maximizar beneficio equivale a minimizar el negativo del retorno: $-\sum_i r_i x_i$.
2. **Minimizamos el riesgo:** El término cuadrático de covarianza ya penaliza pares con alta covarianza: $+\sum_{i < j} c_{ij} x_i x_j$.
3. **Penalizamos la violación del presupuesto:** Convertimos la restricción en un término de penalización cuadrática:
   $$\left( \sum_{i=1}^{n} b_i x_i - B \right)^2$$
   Este término vale $0$ cuando el gasto coincide exactamente con $B$, y penaliza cuadráticamente cualquier desviación.

El Hamiltoniano de coste final unificado queda:
$$\mathcal{H}(x) = -\theta_1 \sum_{i=1}^{n} r_i x_i + \theta_2 \sum_{i=1}^{n}\sum_{j=i+1}^{n} c_{ij} x_i x_j + \theta_3 \left( \sum_{i=1}^{n} b_i x_i - B \right)^2$$

Donde $\theta_1, \theta_2, \theta_3 > 0$ son **multiplicadores de penalización (Lagrange)** que ponderan la importancia relativa asignada a cada objetivo.


In [2]:
import numpy as np

# Retorno esperado de 7 activos
r_i = np.array([77.60417796, 29.71772711, 19.06733564, 32.14568747, 95.12069674, 32.7466954, 56.48788447])

# Coste de inversión de cada activo
b_i = np.array([75, 23, 20, 33, 95, 29, 25])

# Presupuesto disponible: fijamos el 66% de la suma total de costes
B = sum(b_i) * 0.66

# Matriz de covarianza (riesgo conjunto entre activos)
c_ij = np.array([
    [162.22030329,  16.96480901,  53.33392587, -61.19096975, -21.76232416, -93.48033116,  19.59907177],
    [ 16.96480901,  69.33332072,  16.523284  ,  10.15842664, -10.78956293,  24.00854339, -14.78713228],
    [ 53.33392587,  16.523284  ,  43.90334576, -11.43885741, -21.68843841, -22.09673565,  -5.10928035],
    [-61.19096975,  10.15842664, -11.43885741,  46.42412482,  -2.8970541 ,  49.90788498, -31.74061021],
    [-21.76232416, -10.78956293, -21.68843841,  -2.8970541 ,  23.59596426,   4.87211145,  16.74718009],
    [-93.48033116,  24.00854339, -22.09673565,  49.90788498,   4.87211145,  81.35591517, -26.58473057],
    [ 19.59907177, -14.78713228,  -5.10928035, -31.74061021,  16.74718009, -26.58473057,  40.90234169]
])

num_assets = len(r_i)

print(f"Número de activos: {num_assets}")
print(f"Coste total de todos los activos: {sum(b_i)}")
print(f"Presupuesto límite B: {B:.2f}")


Número de activos: 7
Coste total de todos los activos: 300
Presupuesto límite B: 198.00


## 3. Construcción simbólica del modelo con PyQUBO

Escribir las ecuaciones expandidas a mano para construir la matriz $Q$ de tamaño $n \times n$ es propenso a errores. 

La librería `pyqubo` nos permite:
* Declarar arrays de variables binarias con `Array.create(..., vartype='BINARY')`.
* Etiquetar términos con `Constraint(...)` para facilitar el análisis posterior.
* Usar `Placeholder('theta')` para compilar el modelo **una sola vez** y luego poder reajustar los pesos de penalización sin recompilar.


In [3]:
from pyqubo import Array, Placeholder, Constraint

# 1. Definición de variables binarias x_0, x_1, ..., x_6
x = Array.create('x', shape=num_assets, vartype='BINARY')

# 2. Término lineal: Beneficio generado por cada activo
H_linear_profit = 0.0
for i in range(num_assets):
    H_linear_profit += Constraint(r_i[i] * x[i], label=f'profit({i})')

# 3. Término cuadrático: Riesgo obtenido a partir de la matriz de covarianza
H_quadratic = 0.0
for i in range(num_assets):
    for j in range(i + 1, num_assets):
        H_quadratic += Constraint(c_ij[i][j] * x[i] * x[j], label=f'risk({i}, {j})')

# 4. Restricción de presupuesto
H_linear_budget = 0.0
for i in range(num_assets):
    H_linear_budget += Constraint(b_i[i] * x[i], label=f'slot({i})')

# 5. Placeholders para los multiplicadores de Lagrange
theta1 = Placeholder('theta1')
theta2 = Placeholder('theta2')
theta3 = Placeholder('theta3')

# 6. Hamiltoniano unificado
H = - theta1 * H_linear_profit + theta2 * H_quadratic + theta3 * (H_linear_budget - B)**2

# 7. Compilación simbólica
model = H.compile()
print("Modelo compilado con éxito.")


Modelo compilado con éxito.


## 4. Generación de la matriz QUBO

Fijamos los pesos iniciales:
* $\theta_1 = 0.5$ (prioridad al retorno)
* $\theta_2 = 0.3$ (prioridad al control del riesgo)
* $\theta_3 = 0.2$ (prioridad al cumplimiento del presupuesto)

Invocamos `.to_qubo()` pasando el diccionario de valores (`feed_dict`). Esto materializa los coeficientes $Q_{ii}$ (términos lineales) y $Q_{ij}$ (términos cuadráticos), además del `offset` constante.


In [4]:
# Asignación de valores numéricos a los placeholders
feed_dict = {'theta1': 0.5, 'theta2': 0.3, 'theta3': 0.2}

# Obtener la matriz QUBO y el offset constante
qubo, offset = model.to_qubo(feed_dict=feed_dict)

print(f"Offset constante: {offset:.4f}")
print(f"Total de pares (i, j) en el QUBO: {len(qubo)}")
print("\nPrimeros 5 elementos del diccionario QUBO:")
for k, v in list(qubo.items())[:5]:
    print(f"  Q{k} = {v:.4f}")


Offset constante: 7840.8000
Total de pares (i, j) en el QUBO: 28

Primeros 5 elementos del diccionario QUBO:
  Q('x[6]', 'x[2]') = 198.4672
  Q('x[6]', 'x[5]') = 282.0246
  Q('x[4]', 'x[0]') = 2843.4713
  Q('x[4]', 'x[2]') = 753.4935
  Q('x[6]', 'x[6]') = -1883.2439


## 5. Resolución con cálculo exacto (`dimod.ExactSolver`)

Como nuestro espacio de búsqueda tiene sólo $n=7$ activos, el número total de combinaciones posibles es $2^7 = 128$.

`dimod.ExactSolver()` evalúa por fuerza bruta los 128 estados binarios posibles y los ordena por su energía. Esto nos permite **conocer con certeza matemática absoluta el mínimo global** y comprobar la calidad de cualquier método heurístico posterior.


In [5]:
import dimod

# Instanciar el solucionador exacto
exact_sampler = dimod.ExactSolver()

# Muestrear el espacio de estados
response_exact = exact_sampler.sample_qubo(qubo)

# Mostramos las 10 mejores configuraciones encontradas
print("=== Top 10 soluciones con menor energía (ExactSolver) ===")
print(response_exact.slice(10))


=== Top 10 soluciones con menor energía (ExactSolver) ===
  x[0] x[1] x[2] x[3] x[4] x[5] x[6]       energy num_oc.
0    1    1    1    1    0    1    1 -7977.664565       1
1    1    0    0    0    1    1    0 -7976.446948       1
2    0    1    1    1    1    0    1 -7972.776279       1
3    0    0    1    1    1    1    1 -7970.392709       1
4    1    0    0    1    1    0    0 -7963.990385       1
5    0    1    1    0    1    1    1 -7961.841598       1
6    1    1    0    1    0    1    1 -7953.494598       1
7    1    0    0    0    1    0    1 -7949.231201       1
8    0    1    0    1    1    1    1 -7948.440829       1
9    1    1    0    0    1    0    0 -7941.697424       1
['BINARY', 10 rows, 10 samples, 7 variables]


In [6]:
# Extraemos la mejor configuración (menor energía)
best_sample = response_exact.first.sample
solution = [best_sample[f'x[{i}]'] for i in range(num_assets)]

# Calculamos las métricas de negocio para esta solución
effective_profit = np.sum(r_i * solution)
effective_cost = np.sum(b_i * solution)

print("=== Análisis de la Solución Óptima ===")
for i in range(num_assets):
    estado = "Comprar" if solution[i] == 1 else "Ignorar"
    print(f"  Activo {i}: {estado:7s} (Coste: {b_i[i]:2d}, Retorno: {r_i[i]:.2f})")

print(f"\nRetorno total esperado: {effective_profit:.2f}")
print(f"Coste efectivo incurrido: {effective_cost:.2f} (Presupuesto máximo B: {B:.2f})")
print(f"Desviación de presupuesto: {effective_cost - B:+.2f}")


=== Análisis de la Solución Óptima ===
  Activo 0: Comprar (Coste: 75, Retorno: 77.60)
  Activo 1: Comprar (Coste: 23, Retorno: 29.72)
  Activo 2: Comprar (Coste: 20, Retorno: 19.07)
  Activo 3: Comprar (Coste: 33, Retorno: 32.15)
  Activo 4: Ignorar (Coste: 95, Retorno: 95.12)
  Activo 5: Comprar (Coste: 29, Retorno: 32.75)
  Activo 6: Comprar (Coste: 25, Retorno: 56.49)

Retorno total esperado: 247.77
Coste efectivo incurrido: 205.00 (Presupuesto máximo B: 198.00)
Desviación de presupuesto: +7.00


## 6. Resolución heurística con Neal (`SimulatedAnnealingSampler`)

Para problemas reales con decenas o cientos de activos, evaluar $2^N$ estados es computacionalmente intratable para cualquier ordenador clásico.

`neal` es el simulador de **Recocido Simulado** (*Simulated Annealing*) de la suite Ocean de D-Wave. Emula el proceso térmico clásico de enfriamiento en tu propio ordenador (CPU):
* No consume créditos ni requiere token de D-Wave.
* Proporciona la misma interfaz de `dimod` que una QPU real (`DWaveSampler`).
* Es ideal para depurar formulaciones y realizar pruebas de concepto locales.


In [7]:
import neal

# Instanciamos el muestreador de recocido simulado
sa_sampler = neal.SimulatedAnnealingSampler()

# Ejecutamos 100 lecturas independientes del algoritmo estocástico
num_reads = 100
response_sa = sa_sampler.sample_qubo(qubo, num_reads=num_reads)

print("=== Top 10 soluciones obtenidas con Neal (Simulated Annealing) ===")
print(response_sa.slice(10))

best_sa_sample = response_sa.first.sample
sa_solution = [best_sa_sample[f'x[{i}]'] for i in range(num_assets)]
print(f"\n¿Coincide la mejor solución de Neal con la solución exacta? {sa_solution == solution}")
print(f"Energía de Neal: {response_sa.first.energy:.4f} vs Exacta: {response_exact.first.energy:.4f}")


=== Top 10 soluciones obtenidas con Neal (Simulated Annealing) ===
  x[0] x[1] x[2] x[3] x[4] x[5] x[6]       energy num_oc.
0    1    1    1    1    0    1    1 -7977.664565       1
1    1    1    1    1    0    1    1 -7977.664565       1
2    1    1    1    1    0    1    1 -7977.664565       1
3    1    1    1    1    0    1    1 -7977.664565       1
4    1    1    1    1    0    1    1 -7977.664565       1
5    1    1    1    1    0    1    1 -7977.664565       1
6    1    1    1    1    0    1    1 -7977.664565       1
7    1    1    1    1    0    1    1 -7977.664565       1
8    1    1    1    1    0    1    1 -7977.664565       1
9    1    1    1    1    0    1    1 -7977.664565       1
['BINARY', 10 rows, 10 samples, 7 variables]

¿Coincide la mejor solución de Neal con la solución exacta? True
Energía de Neal: -7977.6646 vs Exacta: -7977.6646


## 7. Discusión pedagógica: El ajuste de las penalizaciones ($\theta$)

Observemos un detalle crítico en la solución encontrada:
* El coste efectivo resultante fue **205**, cuando nuestro límite de presupuesto era **198.0** ($+7$ de exceso).

### ¿Por qué ocurrió esto?
En los modelos QUBO, las restricciones **no son barreras rígidas** (*hard constraints*), sino **penalizaciones elásticas** en la función de energía. 

Con $\theta_3 = 0.2$, el optimizador determinó que el beneficio de incluir el activo extra compensaba con creces la penalización impuesta por pasarse 7 unidades de presupuesto:
$$\text{Ganancia de retorno } \theta_1 \cdot r_i > \text{Castigo por violación } \theta_3 \cdot (\Delta B)^2$$

### Experimento: Aumentar la penalización de presupuesto
Veamos qué ocurre si aumentamos $\theta_3$ a $1.0$ y $2.0$ para forzar al solver a respetar estrictamente el presupuesto disponible:


In [8]:
theta3_values = [0.1, 0.2, 0.5, 1.0, 2.0]

print(f"{'theta3':>8s} | {'Energía':>12s} | {'Coste':>8s} | {'Límite B':>8s} | {'¿Factible?':>10s} | {'Activos Seleccionados':>22s}")
print("-" * 80)

for t3 in theta3_values:
    f_dict = {'theta1': 0.5, 'theta2': 0.3, 'theta3': t3}
    q_matrix, _ = model.to_qubo(feed_dict=f_dict)
    
    res = exact_sampler.sample_qubo(q_matrix)
    sol = [res.first.sample[f'x[{i}]'] for i in range(num_assets)]
    c_eff = np.sum(b_i * sol)
    cumple = "SÍ" if c_eff <= B else "NO (Exceso)"
    
    activos_str = str([i for i, val in enumerate(sol) if val == 1])
    print(f"{t3:8.2f} | {res.first.energy:12.2f} | {c_eff:8.1f} | {B:8.1f} | {cumple:>10s} | {activos_str:>22s}")


  theta3 |      Energía |    Coste | Límite B | ¿Factible? |  Activos Seleccionados
--------------------------------------------------------------------------------
    0.10 |     -4062.16 |    205.0 |    198.0 | NO (Exceso) |     [0, 1, 2, 3, 5, 6]
    0.20 |     -7977.66 |    205.0 |    198.0 | NO (Exceso) |     [0, 1, 2, 3, 5, 6]
    0.50 |    -19737.35 |    199.0 |    198.0 | NO (Exceso) |              [0, 4, 5]
    1.00 |    -39338.85 |    199.0 |    198.0 | NO (Exceso) |              [0, 4, 5]
    2.00 |    -78541.85 |    199.0 |    198.0 | NO (Exceso) |              [0, 4, 5]


## 8. Conclusiones y buenas prácticas

1. **Equilibrio de hiperparámetros:** El diseño de la penalización $P$ o $\theta$ es un paso fundamental en computación cuántica de optimización.
   * Si la penalización es **demasiado pequeña**, el sistema preferirá violar las restricciones para obtener más recompensa.
   * Si la penalización es **demasiado grande**, el paisaje energético quedará dominado por las penalizaciones y el solver perderá sensibilidad para optimizar el objetivo real.
2. **Modularidad con PyQUBO:** La posibilidad de compilar una vez y variar los parámetros mediante `Placeholder` permite ajustar los compromisos de forma eficiente.
3. **Flujo de desarrollo local:** Antes de desplegar a una QPU (D-Wave Advantage, etc.), verificar siempre con `ExactSolver` (para problemas pequeños de testeo) y con `neal.SimulatedAnnealingSampler` (para problemas mayores a escala de producción).
